In [4]:
from pathlib import Path
import json
import pickle

import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from tensorflow.keras.preprocessing.sequence import pad_sequences
from transformers import AutoModelForSequenceClassification, AutoTokenizer

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ARTIFACTS_DIR = Path('artifacts')
DEVICE


device(type='cpu')

#### Model Definitions

In [5]:
PAD_IDX = 0


class LogisticRegressionModel(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.linear = nn.Linear(input_dim, 1)

    def forward(self, x):
        return self.linear(x)


class LSTM(nn.Module):
    def __init__(self, embedding_matrix, hidden_dim, num_layers=1, bidirectional=True, dropout=0.3, freeze_embeddings=False):
        super().__init__()

        embedding_tensor = torch.tensor(embedding_matrix, dtype=torch.float32)
        self.embedding = nn.Embedding.from_pretrained(
            embedding_tensor,
            freeze=freeze_embeddings,
            padding_idx=PAD_IDX,
        )
        self.lstm = nn.LSTM(
            input_size=embedding_tensor.shape[1],
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=bidirectional,
            dropout=dropout if num_layers > 1 else 0.0,
        )

        output_dim = hidden_dim * 2 if bidirectional else hidden_dim
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(output_dim, 1)
        self.bidirectional = bidirectional

    def forward(self, input_ids):
        embeddings = self.embedding(input_ids)
        _, (hidden_state, _) = self.lstm(embeddings)

        if self.bidirectional:
            final_hidden = torch.cat((hidden_state[-2], hidden_state[-1]), dim=1)
        else:
            final_hidden = hidden_state[-1]

        logits = self.classifier(self.dropout(final_hidden))
        return logits.squeeze(1)


#### Load Artifact Configs

In [6]:
logistic_dir = ARTIFACTS_DIR / 'logistic_regression'
lstm_dir = ARTIFACTS_DIR / 'lstm_fasttext'
bert_dir = ARTIFACTS_DIR / 'bert'

with (logistic_dir / 'config.json').open('r', encoding='utf-8') as config_file:
    logistic_config = json.load(config_file)

with (lstm_dir / 'config.json').open('r', encoding='utf-8') as config_file:
    lstm_config = json.load(config_file)

bert_config = None
if (bert_dir / 'config.json').exists():
    with (bert_dir / 'config.json').open('r', encoding='utf-8') as config_file:
        bert_config = json.load(config_file)

print('Loaded logistic config:')
print(json.dumps(logistic_config, indent=2))
print('\nLoaded LSTM config:')
print(json.dumps(lstm_config, indent=2))

if bert_config is not None:
    print('\nLoaded BERT config:')
    print(json.dumps(bert_config, indent=2))
else:
    print('\nNo BERT artifacts found yet.')


Loaded logistic config:
{
  "dataset_name": "experiment3",
  "test_size": 0.2,
  "val_size": 0.1,
  "random_seed": 2026,
  "max_features": 5000,
  "batch_size": 128,
  "epochs": 10,
  "learning_rate": 0.001,
  "input_dim": 5000
}

Loaded LSTM config:
{
  "dataset_name": "experiment3",
  "embedding_source": "fasttext",
  "embedding_path": "../data/embeddings/fast_text/crawl-300d-2M-subword/crawl-300d-2M-subword.vec",
  "embedding_dim": 300,
  "max_vocab_size": 30000,
  "max_sequence_length": 128,
  "hidden_dim": 128,
  "num_layers": 1,
  "bidirectional": true,
  "dropout": 0.3,
  "freeze_embeddings": false,
  "batch_size": 256,
  "num_epochs": 100,
  "learning_rate": 0.001,
  "patience": 3,
  "min_delta": 0.001
}

Loaded BERT config:
{
  "dataset_name": "experiment3",
  "model_name": "bert-base-uncased",
  "max_length": 128,
  "batch_size": 256,
  "num_epochs": 3,
  "learning_rate": 0.0003,
  "weight_decay": 0.01,
  "warmup_ratio": 0.1,
  "patience": 2,
  "min_delta": 0.001
}


#### Load Logistic Regression Artifacts

In [7]:
logistic_vectorizer = joblib.load(logistic_dir / 'vectorizer.joblib')
logistic_model = LogisticRegressionModel(logistic_config['input_dim'])
logistic_model.load_state_dict(torch.load(logistic_dir / 'weights.pt', map_location='cpu'))
logistic_model.eval()

print('Loaded logistic regression artifacts.')


Loaded logistic regression artifacts.


#### Load LSTM Artifacts

In [8]:
with (lstm_dir / 'tokenizer.pkl').open('rb') as tokenizer_file:
    lstm_tokenizer = pickle.load(tokenizer_file)

lstm_embedding_matrix = np.load(lstm_dir / 'embedding_matrix.npy')
lstm_model = LSTM(
    embedding_matrix=lstm_embedding_matrix,
    hidden_dim=lstm_config['hidden_dim'],
    num_layers=lstm_config['num_layers'],
    bidirectional=lstm_config['bidirectional'],
    dropout=lstm_config['dropout'],
    freeze_embeddings=lstm_config['freeze_embeddings'],
).to(DEVICE)
lstm_model.load_state_dict(torch.load(lstm_dir / 'weights.pt', map_location=DEVICE))
lstm_model.eval()

print('Loaded LSTM artifacts.')


Loaded LSTM artifacts.


#### Load BERT Artifacts

In [9]:
bert_tokenizer = None
bert_model = None

if bert_config is not None and (bert_dir / 'weights.pt').exists() and (bert_dir / 'tokenizer').exists():
    bert_tokenizer = AutoTokenizer.from_pretrained(str(bert_dir / 'tokenizer'))
    bert_model = AutoModelForSequenceClassification.from_pretrained(bert_config['model_name'], num_labels=2).to(DEVICE)
    bert_model.load_state_dict(torch.load(bert_dir / 'weights.pt', map_location=DEVICE))
    bert_model.eval()
    print('Loaded BERT artifacts.')
else:
    print('Skipping BERT artifact loading because the saved weights/tokenizer were not found.')


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Loaded BERT artifacts.


#### Prediction Helpers

In [10]:
def predict_with_logistic(model, vectorizer, texts):
    if isinstance(texts, str):
        texts = [texts]

    features = vectorizer.transform(texts).toarray()
    feature_tensor = torch.tensor(features, dtype=torch.float32)

    model.eval()
    with torch.no_grad():
        probabilities = torch.sigmoid(model(feature_tensor)).squeeze(1).numpy()

    predictions = (probabilities >= 0.5).astype(int)
    return pd.DataFrame({
        'text': texts,
        'predicted_label': ['AI' if pred == 1 else 'Human' for pred in predictions],
        'ai_probability': probabilities,
    })


def predict_with_lstm(model, tokenizer, texts, max_length):
    if isinstance(texts, str):
        texts = [texts]

    sequences = tokenizer.texts_to_sequences(texts)
    sequences = pad_sequences(
        sequences,
        maxlen=max_length,
        padding='post',
        truncating='post',
    )
    input_tensor = torch.tensor(sequences, dtype=torch.long, device=DEVICE)

    model.eval()
    with torch.no_grad():
        probabilities = torch.sigmoid(model(input_tensor)).cpu().numpy()

    predictions = (probabilities >= 0.5).astype(int)
    return pd.DataFrame({
        'text': texts,
        'predicted_label': ['AI' if pred == 1 else 'Human' for pred in predictions],
        'ai_probability': probabilities,
    })


def predict_with_bert(model, tokenizer, texts, max_length):
    if isinstance(texts, str):
        texts = [texts]

    encodings = tokenizer(
        texts,
        truncation=True,
        padding='max_length',
        max_length=max_length,
        return_tensors='pt',
    )

    input_ids = encodings['input_ids'].to(DEVICE)
    attention_mask = encodings['attention_mask'].to(DEVICE)

    model.eval()
    with torch.no_grad():
        logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
        probabilities = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()

    predictions = (probabilities >= 0.5).astype(int)
    return pd.DataFrame({
        'text': texts,
        'predicted_label': ['AI' if pred == 1 else 'Human' for pred in predictions],
        'ai_probability': probabilities,
    })


#### Run Artifact Predictions

In [11]:
custom_texts = [
    'This essay argues that social media can improve civic participation when paired with strong moderation policies.',
    'In conclusion, the provided evidence clearly demonstrates a multifaceted and highly structured perspective on the topic.',
    "This isn't working very well, as it seems to just be predicting AI for longer messages and Human for shorter messages.",
]

logistic_results = predict_with_logistic(logistic_model, logistic_vectorizer, custom_texts)
lstm_results = predict_with_lstm(lstm_model, lstm_tokenizer, custom_texts, lstm_config['max_sequence_length'])

print('Logistic Regression Predictions')
display(logistic_results)

print('LSTM Predictions')
display(lstm_results)

if bert_model is not None and bert_tokenizer is not None and bert_config is not None:
    bert_results = predict_with_bert(bert_model, bert_tokenizer, custom_texts, bert_config['max_length'])
    print('BERT Predictions')
    display(bert_results)
else:
    print('BERT Predictions skipped because BERT artifacts are not available.')


Logistic Regression Predictions


,text,predicted_label,ai_probability
0,This essay argues that social media can improv...,AI,0.999995
1,"In conclusion, the provided evidence clearly d...",AI,0.995698
2,"This isn't working very well, as it seems to j...",AI,0.884931


LSTM Predictions


,text,predicted_label,ai_probability
0,This essay argues that social media can improv...,AI,0.999987
1,"In conclusion, the provided evidence clearly d...",AI,0.999997
2,"This isn't working very well, as it seems to j...",AI,0.968752


BERT Predictions


,text,predicted_label,ai_probability
0,This essay argues that social media can improv...,AI,0.999793
1,"In conclusion, the provided evidence clearly d...",AI,0.999991
2,"This isn't working very well, as it seems to j...",AI,0.999982
